# Assignment


## Task 3: Transfer Learning

In this task, you will apply transfer learning using a pretrained ResNet50 model (use `ResNet50_Weights.IMAGENET1K_V2`) trained on ImageNet.
https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet50.html

Transfer learning is a powerful machine learning technique for leveraging knowledge learned from one task and applying it to a new task. For instance, if a model has been trained to recognize raccoons, the features it has learned (such as shapes, textures, or patterns) can be useful for identifying tanukis, animals with similar visual characteristics. This approach is particularly valuable when the available dataset for the new task is limited in size, making it difficult to train a full-scale model from scratch. The typical workflow for transfer learning in deep learning involves several key steps:

1. **Starting with a pretrained model:** The architecture of a model trained on a large dataset is (partially) replicated, and the weights of some of its layers are imported.
2. **Freezing the pretrained layers:** These layers are frozen (i.e., excluded from the backpropagation update) to prevent the information learned from the initial task from being overwritten during the new training process.
3. **Adding new trainable layers:** These layers are introduced on top of the frozen ones and are specifically designed to adapt the previous knowledge to the new task.
4. **Training on the new dataset:** The model with the new layers is then trained using data from the current task.
5. **Fine-tuning (optional):** In some cases, the entire model (or parts of it) is unfrozen and retrained with a low learning rate. This fine-tuning allows the model to adapt more precisely to the new dataset, often resulting in improved performance.

You will use pretrained features from models trained on a large-scale image recognition dataset, study the effect of transfer learning on model performance and training time, and compare it to the baseline CNN model you trained from scratch in the previous section. You can find an example
of transfer learning here:
https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html

Create a new notebook `transferlearning.ipynb`. Again, copy the dataset logic from the first task.

### 1. Set up the classification

- Choose the same metric(s) for this task as you did in the previous one.

### 2. Set up the base model

- Instantiate the model with the ImageNet pretrained weights. Do not include the final fully connected classification layer. _(Note: this is the last layer of the network.)_
- Freeze all layers in the base model.
- Add one or more fully connected layers on top of the base model to fit the classification task.
- Add one dropout layer before the last layer(s). Optionally, add more.
- Print the final architecture.

### 3. Train your model

- For the initial model, start with the best hyperparameters found while optimizing the baseline (when applicable).
- Train your initial model using the previously selected loss function and evaluate the training and validation losses.
- Evaluate the model using the same performance metric(s) chosen in Task 2.
- Ensure that the model runs without errors and that the loss decreases more or less smoothly.
- Plot the training and validation curves.
- Again, tune your hyperparameters in a systematic way, but you do not need to perform a full-scale analysis.

### 4. Train your model on the full dataset

- Retrain your model using the complete training dataset (including the validation dataset), using your previous set of optimal hyperparameters.
- Plot the training curve.
- Save and reload your model.

### 5. Fine-tuning the entire model

- Unfreeze all or part of the base model and train the whole model end-to-end for the same number of epochs as you trained the pretrained model.
- If you notice a sudden drop in performance on the training dataset, reload the model and try again with a smaller learning rate. If there is no sign of improvement, skip this step and use the saved model from the previous step in the following parts. Save the best model.
- Test the saved model on the test dataset and print the performance metric(s) you chose for this problem.
- Display the confusion matrix.
- Plot a few samples from the test dataset (without preprocessing) with their predictions (after preprocessing).

---

In your report, describe the transfer learning you performed, including the following:

- **Model:** How did you choose the architecture for the final layers you added to the pretrained model?
- **Performance:** How did the pretrained model's training curve compare to the final baseline model?
- **Fine-tuning:** Describe any hyperparameters you changed or further fine-tuning you performed.
- **Transfer Learning:** Discuss, based on this experience, the advantages and disadvantages of this approach.


# Solution


Sources

- https://docs.pytorch.org/vision/main/models.html
- https://docs.pytorch.org/vision/main/models.html#using-the-pre-trained-models (resnet50 trained on images with specific transforms!)


## 0. Setup


Note: Torchvision not in original list of libraries used in the course


In [2]:
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn

# import torchvision
from torchvision import datasets, models, transforms

## 1. Download and explore pretrained model


Download model from torchvision.models

- Weights version determined by the assignment


In [7]:
tf_resnet = models.resnet50(weights="IMAGENET1K_V2")

### Explore model architecture


#### Inspect architecture, shape of layers


In [13]:
print("Model's state_dict:")
for param_tensor in tf_resnet.state_dict():
    print(param_tensor, "\t", tf_resnet.state_dict()[param_tensor].size())

Model's state_dict:
conv1.weight 	 torch.Size([64, 3, 7, 7])
bn1.weight 	 torch.Size([64])
bn1.bias 	 torch.Size([64])
bn1.running_mean 	 torch.Size([64])
bn1.running_var 	 torch.Size([64])
bn1.num_batches_tracked 	 torch.Size([])
layer1.0.conv1.weight 	 torch.Size([64, 64, 1, 1])
layer1.0.bn1.weight 	 torch.Size([64])
layer1.0.bn1.bias 	 torch.Size([64])
layer1.0.bn1.running_mean 	 torch.Size([64])
layer1.0.bn1.running_var 	 torch.Size([64])
layer1.0.bn1.num_batches_tracked 	 torch.Size([])
layer1.0.conv2.weight 	 torch.Size([64, 64, 3, 3])
layer1.0.bn2.weight 	 torch.Size([64])
layer1.0.bn2.bias 	 torch.Size([64])
layer1.0.bn2.running_mean 	 torch.Size([64])
layer1.0.bn2.running_var 	 torch.Size([64])
layer1.0.bn2.num_batches_tracked 	 torch.Size([])
layer1.0.conv3.weight 	 torch.Size([256, 64, 1, 1])
layer1.0.bn3.weight 	 torch.Size([256])
layer1.0.bn3.bias 	 torch.Size([256])
layer1.0.bn3.running_mean 	 torch.Size([256])
layer1.0.bn3.running_var 	 torch.Size([256])
layer1.0.bn3.num

#### Input expected?


In [ ]:
weights = models.ResNet50_Weights.IMAGENET1K_V2

# AI suggested but not very interesting (except maybe reference to training recipe?)
# print(weights.meta)

In [ ]:
print(weights.transforms())

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


AI-generated ↓

Type: RGB images
Channels: 3 (Red, Green, Blue)
Spatial size: Typically 224 × 224
Tensor shape: (batch_size, 3, 224, 224)
Data type: torch.float32
Value range: Normalized (not raw 0–255 pixels)


Shape of weights of first layer: (out_channels, in_channels, kernel_height, kernel_width)


In [ ]:
tf_resnet.state_dict()["conv1.weight"].shape

torch.Size([64, 3, 7, 7])

This is the first layer of the model.

- 3 channels
- image size must be at least 7x7, in practice should obviously be much larger
- 64 filters: determine what kinds of basic things exist (in 2d images: basic shapes). The model is learning which 64 fundamental visual features it should care about. Why 64?
  - feature diversity: 8 "basic things" would be too limiting
  - trade off between performance and capacity (memory, compute)


In [19]:
tf_resnet.conv1

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

Each one of the 64 filters produces a feature map that is passed to the next layer. Each feature map has shape 112x112 (depends on 224x224 size of each channel + stride + padding).


Transforms on images

- Zie https://docs.pytorch.org/vision/main/models.html#using-the-pre-trained-models


In [ ]:
weights = models.ResNet50_Weights.IMAGENET1K_V2
preprocess = weights.transforms()
preprocess
# # Apply it to the input image
# img_transformed = preprocess(img)

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

#### Output given?


Final fully connected layer


In [ ]:
tf_resnet.fc.out_features

#### Feed random data to model as test


In [34]:
# Equates to one random 224x224 image with 3 channels
random_data = torch.rand((1, 3, 224, 224))  # value for batch size is required (1)

In [ ]:
# Pass through model
result = tf_resnet(random_data)
result

As expected, 1000 output values, 1 for each output class.

Output is meaningless as random_data is nothing like Imagenet data, which has gone through specific transforms.


In [ ]:
# # Typical preprocessing for resnet50
# preprocess = transforms.Compose(
#     [
#         transforms.ToTensor(),  # scales to [0, 1]
#         transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#     ]
# )

In [ ]:
# Mimic preprocessing for random_data --> distribution of random_data is similar to data that resnet was trained on
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

random_data_pp = (random_data - mean) / std

In [ ]:
result_pp = tf_resnet(random_data_pp)
result_pp

#### Conclusions


Two conclusions

1. input expected by resnet50 is 224x224 image with 3 channels. Data used for baseline model in task 2 was 128x128 (downsampled from 224x224) with 1 channel
2. too many output classes (1000) compared to our classification task (2 classes)


Two solutions

1. Turning 1 channel into 3 channels not difficult, but what about comparison to baseline model from task 2? Appels and oranges? Alternative solution: adapt resnet to accept 1-channel image (mean of 3 channels). Compare both approaches?
2. Replace final fully connected layer (AFTER freezing weights!)


In [ ]:
# Input layer

### Adapt dataset to Resnet


PIL-based approach (single image)


In [ ]:
weights = models.ResNet50_Weights.IMAGENET1K_V2
img_path = "../../data/1_source/test/0/9003175L.png"

img = Image.open(img_path).convert("RGB")  # force 1 channel to 3 channels
x = weights.transforms()(img).unsqueeze(0)  # add value for batch size

Tensor-based approach (single image)

- useful here? When using Dataset & DataLoader workflow there is no need for explicit conversion of images to tensors


In [ ]:
# AI-generated ↓

# grayscale tensor: (1, H, W)
gray = torch.randn(1, 224, 224)

# repeat channels → (3, H, W)
rgb_like = gray.repeat(3, 1, 1)
rgb_like

Note : choose approach consistent with tasks 1 & 2


### Adapt Resnet to dataset


In [ ]:
# AI-generated ↓

import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)

# Replace first conv layer
old_conv = model.conv1
model.conv1 = nn.Conv2d(
    in_channels=1, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False
)

# Initialize new conv weights using the old ones
with torch.no_grad():
    model.conv1.weight[:] = old_conv.weight.mean(dim=1, keepdim=True)

## 2. Freeze the pretrained layers


**Voor** het vervangen van de final fully connected layer! Anders worden de weights van die nieuwe layer ook bevroren!


In [ ]:
for param in tf_resnet.parameters():
    param.requires_grad = False

## 3. Add new trainable layers

- https://discuss.pytorch.org/t/add-layers-on-pretrained-model/88760
- https://stackoverflow.com/questions/44406819/pytorch-custom-layer-is-not-a-module-subclass


### Replace final fully connected layer


In [ ]:
# Too many output features (1000)
tf_resnet.fc

Linear(in_features=2048, out_features=1000, bias=True)

In [ ]:
class_names = [0, 1]
num_features = tf_resnet.fc.in_features
tf_resnet.fc = nn.Linear(in_features=num_features, out_features=len(class_names)) # klopt dit wel? out_features is hier 2?

Because this layer has been added after the freezing of the pretrained weights, it has (by default) `requires_grad = True` and will be trained during backpropagation.


### Add other layers

Nu: hoe?
Later beslissen/verantwoorden: waarom?


Uit assignment: "How did you choose the architecture for the final layers you added to the pretrained model?"

- dropout layer to avoid overfitting
- batch normalization to stabilize training
- ReLu for non-linearity


In [ ]:
# AI-generated ↓
model.fc = nn.Sequential(
    nn.Linear(num_features, 256),  # intermediate layer
    nn.ReLU(),  # non-linearity
    nn.Dropout(p=0.5),  # regularization to reduce overfitting
    nn.Linear(256, 1),  # final binary output
)

## 4. Train on the new dataset


Define loss function and optimizer

Nu: hoe?
Later: keuzes maken en verantwoorden


Uit assignment: "For the initial model, start with the best hyperparameters found while optimizing the
baseline (when applicable)."

M.a.w. learning rate, loss function etc. overnemen van taak 2


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD()

Training loop skeleton


In [ ]:
# AI generated ↓

tf_resnet.train()

for epoch in range(num_epochs):
    for images, labels in dataloader:
        labels = labels.float().unsqueeze(1)  # shape: (batch, 1)
        optimizer.zero_grad()  # clear gradients from previous step
        outputs = tf_resnet(images)  # forward pass → shape: (batch, 1)
        loss = criterion(outputs, labels)
        loss.backward()  # compute gradients
        optimizer.step()  # update weights

## 5. Finetune (optional)


## 6. Inference


Do not forget:

1. resnet50 expects specific data transformations to be performed on images
2. set to evaluation mode: tf_resnet.eval()


In [ ]:
tf_resnet.eval()

with torch.no_grad():
    outputs = tf_resnet(images)  # raw logits
    probs = torch.sigmoid(outputs)  # convert to probabilities in [0, 1]
    preds = (probs >= 0.5).long()  # threshold at 0.5 → 0 or 1

Hierboven: AI-generated code

- andere threshold probability gebruiken (cfr. vermijden van false negatives)


In [ ]:
tf_resnet.eval()

with torch.no_grad():
    outputs = tf_resnet(images)  # raw logits
    probs = torch.sigmoid(outputs)  # convert to probabilities in [0, 1]
    preds = (probs >= 0.5).long()  # threshold at 0.5 → 0 or 1

## 7. Model evaluation metrics
